# Monitoring a deployed system for drift with multiple judges

**`AsymptoticMultiPPRM`** and its label-only counterpart **`AsymptoticClassicalRM`** watch a quality metric across successive batches of production data and raise an alarm the moment there is statistically valid evidence that it has drifted past a threshold you fix in advance. This is necessary because a deployed system's quality is rarely static: traffic shifts and upstream models change, so a metric that looked fine at launch can drift silently, and a single-shot estimate made at first deployment time cannot catch that.

---

**What you will learn:**

- Why re-checking a fresh confidence interval multiple times inflates the false-alarm rate
- How `AsymptoticMultiPPRM` and `AsymptoticClassicalRM` give an anytime-valid alternative that stays safe under repeated checks
- How combining multiple correlated proxy judges lets `AsymptoticMultiPPRM` detect a drift sooner than the label-only `AsymptoticClassicalRM`
- How to read their running-mean curves, confidence bounds, and alarms

## The problem: the metric can drift after launch, and repeated checking breaks validity

Suppose you run the same customer-facing AI assistant handling an open-ended troubleshooting flow, hallucinations are a real risk there, and you rely on two LLM judges, Gemini and Claude, to measure the hallucination rate. Every week, a fresh batch of conversations is judged by both, and a handful are sent for human review. Your team has fixed a business threshold on the hallucination rate, and wants to know as soon as the rate has drifted past it since launch.

The tempting approach is to compute a fresh confidence interval every week and raise an alarm the moment one crosses the threshold. This is **peeking**: each weekly test carries its own false-alarm probability, and checking after every batch compounds those chances, so a false alarm becomes likely over a long enough monitoring horizon even if nothing has actually drifted. The [Monitors user guide](../../../user_guide/monitors/) explains why this happens and how an anytime-valid confidence sequence that stays safe no matter how often you look avoids it. This tutorial focuses on what that means for you in practice.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

from glide.estimators import MultiPPIMeanEstimator
from glide.monitors import AsymptoticClassicalRM, AsymptoticMultiPPRM
from glide.samplers import UniformSampler
from glide.simulators import generate_stratified_multi_binary_dataset, simulate_annotation

# ── Colour palette ──────────────────────────────────────────
C_ESTIMATE = "#95A5A6"  # per-batch point estimate              — grey
C_NAIVE = "#E74C3C"  # naive baseline alarm                     — red-orange
C_PPRM = "#27AE60"  # Multi-PPRM running mean/bound              — green
C_CLASSICAL = "#2980B9"  # classical monitor running mean/bound — blue
C_ALARM_PPRM = "#8E44AD"  # Multi-PPRM alarm                     — purple
C_ALARM_CLASSICAL = "#F39C12"  # classical monitor alarm        — gold
C_THRESHOLD = "#2C3E50"  # threshold                             — dark slate

# ── Global plot style ────────────────────────────
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "#FAFAFA",
        "axes.grid": True,
        "grid.color": "#E5E5E5",
        "grid.linewidth": 0.8,
        "font.size": 18,
        "axes.labelsize": 18,
        "axes.titlesize": 18,
        "legend.fontsize": 14,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        "figure.titlesize": 19,
    }
)

## Simulating a batched production stream

We use `generate_stratified_multi_binary_dataset` to simulate successive batches of conversations, each with its own true hallucination rate, judged throughout by both Gemini and Claude, stacked into a single stream with an integer identifying the chronological batch each row belongs to. Gemini and Claude each keep a fixed correlation with the true label, `CORRELATIONS`, held constant batch after batch via `np.tile`. Their own reported rate, however, is not pinned: each carries a fixed additive bias above the true rate, so `proxy_means = true_means[:, np.newaxis] + BIASES` tracks `true_means` as it drifts, the same way the single-proxy tutorial's `proxy_means = true_means + PROXY_BIAS` does.

`simulate_batches` below builds such a stream from a list of per-week true hallucination rates. Every week gets `BATCH_SIZE` conversations judged by both Gemini and Claude, of which `LABELS_PER_BATCH` are sent for human review: we use the `UniformSampler` to simulate uniform allocation of the review budget at each week. The human labels are revealed only for selected conversations, leaving the rest as `np.nan`. We fix an alarm threshold above the stationary rate and assess the monitoring methods' ability to detect when it is exceeded while avoiding false alarms. All batches in the stream are monitored.

In [ ]:
N_BATCHES = 50
BATCH_SIZE = 800
LABELS_PER_BATCH = 40
STATIONARY_MEAN = 0.28
BIASES = np.array([0.05, 0.08])
CORRELATIONS = [0.70, 0.80]
CONFIDENCE_LEVEL = 0.8
THRESHOLD = 0.30


def simulate_batches(true_means, random_seed):
    n_batches = len(true_means)
    proxy_means = true_means[:, np.newaxis] + BIASES
    y_true_oracle, y_proxies, batches = generate_stratified_multi_binary_dataset(
        n_samples=np.full(n_batches, BATCH_SIZE),
        true_mean=true_means,
        proxy_means=proxy_means,
        correlations=np.tile(CORRELATIONS, (n_batches, 1)),
        random_seed=random_seed,
    )
    xi = np.hstack(
        [UniformSampler().sample(BATCH_SIZE, LABELS_PER_BATCH, random_seed=random_seed + i) for i in range(n_batches)]
    )
    y_true = simulate_annotation(y_true_oracle, xi)
    return y_true, y_proxies, batches

## The naive baseline: one confidence interval per batch

The tempting-but-invalid approach simply calls `MultiPPIMeanEstimator` once per week, comparing that week's confidence interval to the threshold and alarming whenever the interval lies above it.

In [ ]:
def naive_per_batch_alarms(y_true, y_proxies, batches, threshold, n_batches):
    means = np.zeros(n_batches)
    lower_bounds = np.zeros(n_batches)
    alarms = np.zeros(n_batches, dtype=bool)
    for batch_id in range(n_batches):
        batch_mask = batches == batch_id
        batch_result = MultiPPIMeanEstimator().estimate(y_true[batch_mask], y_proxies[batch_mask])
        means[batch_id] = batch_result.mean
        lower_bounds[batch_id] = batch_result.confidence_interval.lower_bound
        alarms[batch_id] = batch_result.confidence_interval.lower_bound > threshold
    return means, lower_bounds, alarms

In [ ]:
def plot_monitor_vs_baseline(lower_bounds, multi_pprm_result, classical_result, naive_alarms, title):
    weeks = np.arange(1, len(lower_bounds) + 1)
    fig, ax = plt.subplots(figsize=(10, 5.5))

    ax.scatter(weeks, lower_bounds, color=C_ESTIMATE, s=28, alpha=0.8, zorder=3, label="Per-batch lower bounds")
    ax.plot(
        weeks,
        multi_pprm_result.confidence_bounds,
        color=C_PPRM,
        linewidth=2.5,
        zorder=4,
        label="Multi-PPRM lower bounds",
    )
    ax.plot(
        weeks,
        classical_result.confidence_bounds,
        color=C_CLASSICAL,
        linewidth=2.5,
        zorder=4,
        label="Classical monitor lower bounds",
    )
    ax.axhline(THRESHOLD, color=C_THRESHOLD, linestyle="--", linewidth=1.8, zorder=2, label="Threshold")

    if naive_alarms.any():
        ax.scatter(
            weeks[naive_alarms],
            lower_bounds[naive_alarms],
            marker="+",
            s=110,
            linewidths=2.5,
            color=C_NAIVE,
            zorder=5,
            label="Naive baseline alarms",
        )

    if multi_pprm_result.drift_detected:
        alarm_week = multi_pprm_result.first_alarm_index + 1
        ax.axvline(
            alarm_week,
            color=C_ALARM_PPRM,
            linestyle=":",
            linewidth=2.2,
            zorder=2,
            label=f"Multi-PPRM alarm (week {alarm_week})",
        )

    if classical_result.drift_detected:
        alarm_week = classical_result.first_alarm_index + 1
        ax.axvline(
            alarm_week,
            color=C_ALARM_CLASSICAL,
            linestyle=":",
            linewidth=2.2,
            zorder=2,
            label=f"Classical alarm (week {alarm_week})",
        )

    ax.set_xlabel("Week")
    ax.set_ylabel("Hallucination Rate")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.set_title(title)
    ax.set_ylim(-0.05, 1.0)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(loc="upper left", framealpha=0.9)
    plt.tight_layout()
    plt.show()

## Scenario 1: no drift, no false alarm

We generate a fully stationary stream: the true hallucination rate stays below the threshold. Neither method should alarm, except by chance.

In [ ]:
true_means_no_drift = np.full(N_BATCHES, STATIONARY_MEAN)
y_true_no_drift, y_proxies_no_drift, batches_no_drift = simulate_batches(true_means_no_drift, random_seed=1)

n_labeled = np.sum(~np.isnan(y_true_no_drift))
print(f"Total conversations : {len(y_true_no_drift):,}")
print(f"  Human-reviewed     : {n_labeled:,}")
print(f"  Number of weeks    : {len(np.unique(batches_no_drift))}")

In [ ]:
n_nan_in_y_true = np.isnan(y_true_no_drift).sum()
n_zeros_in_y_true = np.sum(y_true_no_drift == 0)
n_ones_in_y_true = np.sum(y_true_no_drift == 1)
print("Analysis of the human-reviewed labels:")
print(f"Number of NaNs: {n_nan_in_y_true}")
print(f"Number of 0s: {n_zeros_in_y_true}")
print(f"Number of 1s: {n_ones_in_y_true}")

In [ ]:
y_gemini_no_drift = y_proxies_no_drift[:, 0]
y_claude_no_drift = y_proxies_no_drift[:, 1]

for judge_name, y_judge in [("Gemini", y_gemini_no_drift), ("Claude", y_claude_no_drift)]:
    print(f"Analysis of the {judge_name} labels:")
    print(f"Number of NaNs: {np.isnan(y_judge).sum()}")
    print(f"Number of 0s: {np.sum(y_judge == 0)}")
    print(f"Number of 1s: {np.sum(y_judge == 1)}")

In [ ]:
counts = np.bincount(batches_no_drift.astype(int))
print(f"There are {len(counts)} batches, with the following number of conversations per batch:")
print(counts)

Run the naive baseline and both monitoring methods.

In [ ]:
means_no_drift, lower_no_drift, naive_alarms_no_drift = naive_per_batch_alarms(
    y_true_no_drift, y_proxies_no_drift, batches_no_drift, THRESHOLD, N_BATCHES
)
multi_pprm_result_no_drift = AsymptoticMultiPPRM().detect(
    y_true_no_drift,
    y_proxies_no_drift,
    batches_no_drift,
    higher_is_better=False,
    threshold=THRESHOLD,
    confidence_level=CONFIDENCE_LEVEL,
    metric_name="Hallucination Rate",
)
classical_result_no_drift = AsymptoticClassicalRM().detect(
    y_true_no_drift,
    batches_no_drift,
    higher_is_better=False,
    threshold=THRESHOLD,
    confidence_level=CONFIDENCE_LEVEL,
    metric_name="Hallucination Rate",
)
naive_alarm_weeks = np.arange(1, N_BATCHES + 1)[naive_alarms_no_drift]

print(f"Naive baseline alarms at week(s): {naive_alarm_weeks}")
print(f"Multi-PPRM drift detected: {multi_pprm_result_no_drift.drift_detected}")
print(f"Classical monitor drift detected: {classical_result_no_drift.drift_detected}")
print(
    f"Multi-PPRM running mean / bound after {N_BATCHES} weeks: "
    f"{multi_pprm_result_no_drift.running_means[-1]:.1%} / {multi_pprm_result_no_drift.confidence_bounds[-1]:.1%}"
)
print(
    f"Classical monitor running mean / bound after {N_BATCHES} weeks: "
    f"{classical_result_no_drift.running_means[-1]:.1%} / {classical_result_no_drift.confidence_bounds[-1]:.1%}"
)

In [ ]:
plot_monitor_vs_baseline(
    lower_no_drift,
    multi_pprm_result_no_drift,
    classical_result_no_drift,
    naive_alarms_no_drift,
    title="Scenario 1: stationary stream (no drift)",
)

At some point, ordinary sampling noise pushes one of the confidence intervals above the threshold, and the naive baseline raises a false alarm, purely from **peeking**: with a fresh, independent-looking test every week, the chance of at least one false alarm climbs the longer you keep checking, even though the hallucination rate did not move. `AsymptoticMultiPPRM` and `AsymptoticClassicalRM`, in contrast, never alarm across the full stream: their running means stay below the threshold, with anytime-valid lower bounds safely underneath it as well.

## Scenario 2: slow drift, early alarm

Now we generate data with a progressive drift: partway through the stream, the hallucination rate ramps up to an unacceptably high level.

In [ ]:
DRIFT_START = 15
DRIFT_AMPLITUDE = 0.35
RAMP_LENGTH = 30

true_means_slow_drift = np.hstack(
    [
        np.full(DRIFT_START, STATIONARY_MEAN),
        np.linspace(STATIONARY_MEAN, STATIONARY_MEAN + DRIFT_AMPLITUDE, RAMP_LENGTH),
        np.full(N_BATCHES - RAMP_LENGTH - DRIFT_START, STATIONARY_MEAN + DRIFT_AMPLITUDE),
    ]
)
y_true_slow_drift, y_proxies_slow_drift, batches_slow_drift = simulate_batches(true_means_slow_drift, random_seed=1)

means_slow_drift, lower_slow_drift, naive_alarms_slow_drift = naive_per_batch_alarms(
    y_true_slow_drift, y_proxies_slow_drift, batches_slow_drift, THRESHOLD, N_BATCHES
)
multi_pprm_result_slow_drift = AsymptoticMultiPPRM().detect(
    y_true_slow_drift,
    y_proxies_slow_drift,
    batches_slow_drift,
    higher_is_better=False,
    threshold=THRESHOLD,
    confidence_level=CONFIDENCE_LEVEL,
    metric_name="Hallucination Rate",
)
classical_result_slow_drift = AsymptoticClassicalRM().detect(
    y_true_slow_drift,
    batches_slow_drift,
    higher_is_better=False,
    threshold=THRESHOLD,
    confidence_level=CONFIDENCE_LEVEL,
    metric_name="Hallucination Rate",
)

naive_alarm_weeks = np.flatnonzero(naive_alarms_slow_drift) + 1
print(f"Naive baseline alarms: {len(naive_alarm_weeks)} weeks, first at week {naive_alarm_weeks[0]}")
multi_pprm_alarm_index = multi_pprm_result_slow_drift.first_alarm_index
if multi_pprm_alarm_index is not None:
    print(f"Multi-PPRM drift detected: True, first alarm at week {multi_pprm_alarm_index + 1}")
classical_alarm_index = classical_result_slow_drift.first_alarm_index
if classical_alarm_index is not None:
    print(f"Classical monitor drift detected: True, first alarm at week {classical_alarm_index + 1}")

In [ ]:
plot_monitor_vs_baseline(
    lower_slow_drift,
    multi_pprm_result_slow_drift,
    classical_result_slow_drift,
    naive_alarms_slow_drift,
    title="Scenario 2: drift occurrence during deployment",
)

Observe that the peeking baseline alarms earlier. In real time, however, a team watching it has no way to tell this alarm apart from a lucky (or unlucky) one, because its error rate was never controlled for repeated testing in the first place: Scenario 1 already showed the exact same procedure crying wolf on a stream that never drifted at all. The anytime-valid alarms carry no such ambiguity, since each is backed by the same guarantee demonstrated in Scenario 1: under no drift, the chance either would ever fire at all across the whole monitoring horizon corresponds to the confidence level used, so at the moment each fires here, that guarantee makes the alarm trustworthy without waiting for later weeks to confirm it.

Between the two anytime-valid monitors, `AsymptoticMultiPPRM` alarms sooner than `AsymptoticClassicalRM`: combining Gemini's and Claude's correlated judgments sharpens its per-batch estimate on top of the human labels both monitors share, shrinking its confidence sequence and letting it accumulate evidence of the drift faster. By the end of the stream, both running means and their bounds are clearly above the threshold.

## Reading the result object

In both scenarios, `AsymptoticMultiPPRM` returned a `PredictionPoweredMeanMonitoringResult`. Its public surface covers everything used above, plus a formatted summary.

In [ ]:
print("Multi-PPRM")
print(f"{'Drift detected':<32}: {multi_pprm_result_slow_drift.drift_detected}")
print(f"{'First alarm index':<32}: {multi_pprm_result_slow_drift.first_alarm_index}")
print(f"{'Alarm threshold':<32}: {multi_pprm_result_slow_drift.alarm_threshold}")
print(f"{'Confidence level':<32}: {multi_pprm_result_slow_drift.confidence_level}")
print(f"{'Running means (last 3)':<32}: {np.round(multi_pprm_result_slow_drift.running_means[-3:], 3)}")
print(f"{'Confidence bounds (last 3)':<32}: {np.round(multi_pprm_result_slow_drift.confidence_bounds[-3:], 3)}")
print(f"{'True labels per batch (last 3)':<32}: {multi_pprm_result_slow_drift.batch_n_true[-3:]}")
print(f"{'Proxy rows per batch (last 3)':<32}: {multi_pprm_result_slow_drift.batch_n_proxy[-3:]}")

`batch_n_proxy` counts the shared row count of `y_proxies`: the number of conversations judged in each batch, not the number of judges. It is unaffected by the number of proxies involved, two here for Gemini and Claude together, exactly as it would be for a single judge.

## Summary: Anytime-valid Multi-Proxy Risk Monitoring detects drifts and reduces false alarms

| | Per-batch baseline | `AsymptoticClassicalRM` | `AsymptoticMultiPPRM` |
|-|---------------------|-------------------|-------------------|
| Valid under repeated looks? | ❌ No | ✅ Yes | ✅ Yes |
| Uses proxy labels? | ✅ Yes, from both judges | ❌ No | ✅ Yes, from both judges |
| Detects a sustained drift? | 🟠 Yes, but with no error-rate guarantee | ✅ Yes | ✅ Yes, sooner than the classical monitor |

**Key takeaways:**

1. **Repeated significance testing is not a monitor.** Scenario 1 showed the per-batch baseline raise a false alarm on a perfectly stationary stream, purely from checking every week.
2. **An anytime-valid confidence sequence makes checking every week safe.** Both `AsymptoticMultiPPRM` and `AsymptoticClassicalRM`'s guarantees hold simultaneously across all looks, so their alarms in Scenario 2 can be trusted the moment they fire, without waiting to see whether later weeks confirm them.
3. **Combining proxies, not one stronger judge, is what speeds up detection here.** `AsymptoticMultiPPRM` alarmed sooner than `AsymptoticClassicalRM` in Scenario 2 by pooling Gemini's and Claude's correlated labels on top of the human labels both monitors share; the gain comes from combining both judges together, not from either one individually being the stronger of the two.
4. **The naive baseline's alarms carry no error-rate guarantee, correct or not.** Scenario 2's naive alarms happened to line up with a real regression, but Scenario 1 already showed the same procedure crying wolf under no drift at all: there is no way to tell the two situations apart from the naive alarms alone.

---

*Want to go further? The [Monitors user guide](../../../user_guide/monitors/) derives the anytime-valid confidence sequence and its multi-proxy extension used here from first principles.*